# Named Entity Recognition (NER)

## What is NER?
NER identifies and classifies named entities in text into predefined categories:
- **PER**: Person names *"Barack Obama"*
- **ORG**: Organizations *"Google"*, *"United Nations"*
- **LOC**: Locations *"Paris"*, *"Eiffel Tower"*
- **DATE**: Temporal expressions *"January 2024"*
- **MONEY**: Monetary values *"$1 billion"*
- **MISC**: Miscellaneous products, events, nationalities

**Applications**: Information extraction, knowledge graph construction, question answering, document understanding.

---

## 1. Tagging Schemes

### IOB (Inside-Outside-Beginning):
```
Barack  Obama  visited  New   York   City
B-PER   I-PER  O        B-LOC I-LOC  I-LOC
```

### BIOES (Begin-Inside-Outside-End-Single):
```
Barack  Obama  visited  New   York   City
B-PER   E-PER  O        B-LOC I-LOC  E-LOC
```
BIOES is more expressive single-token entities get tag `S-TYPE`.

---

## 2. CRF (Conditional Random Field)

CRF models the conditional probability of the label sequence given the input:

$$P(y|x) = \frac{1}{Z(x)} \exp\left( \sum_{t=1}^{T} \sum_k \lambda_k f_k(y_{t-1}, y_t, x, t) \right)$$

Where:
- $f_k(y_{t-1}, y_t, x, t)$ = feature functions (e.g., current word is capitalized, previous label is B-PER)
- $\lambda_k$ = learned feature weights
- $Z(x)$ = partition function (normalization)

CRF captures **label dependencies** e.g., `I-PER` can only follow `B-PER` or `I-PER`.

---

## 3. BiLSTM-CRF

State-of-the-art pre-transformer NER architecture:

1. **Embedding layer**: Word embeddings + character embeddings
2. **BiLSTM**: Captures context from both directions
3. **CRF layer**: Enforces valid tag sequences

$$h_t = [\overrightarrow{LSTM}(x_1,...,x_t); \overleftarrow{LSTM}(x_T,...,x_t)]$$

$$P(y|x) = \text{CRF}(h_1, ..., h_T)$$

---

## 4. BERT-based NER

Fine-tune BERT for token classification. Each token's hidden state predicts its label:

$$P(y_i|x) = \text{softmax}(W \cdot h_i^{(L)})$$

**Subword alignment**: BERT uses subword tokenization. Map subword labels back to word labels (usually take first subword prediction).

---

## 5. Relation Extraction

After NER: extract **relationships** between entities.

Example: *"Elon Musk founded Tesla"* → `(Elon Musk, FOUNDED, Tesla)`

Methods:
- Rule-based (dependency patterns)
- Supervised classification (given entity pair, predict relation type)
- Joint NER + RE models
- LLM-based extraction

In [1]:
import spacy
import numpy as np

# ============================================================
# NER WITH spaCy
# ============================================================
try:
    nlp = spacy.load('en_core_web_sm')
    
    texts = [
        "Apple Inc. was founded by Steve Jobs, Steve Wozniak, and Ronald Wayne in Cupertino, California on April 1, 1976.",
        "The United Nations headquarters is located in New York City. António Guterres serves as Secretary-General.",
        "Tesla reported revenue of $25.1 billion in Q3 2023. Elon Musk is the CEO."
    ]
    
    for text in texts:
        doc = nlp(text)
        print(f"Text: {text[:60]}...")
        print("Entities:")
        for ent in doc.ents:
            print(f"  '{ent.text}' → {ent.label_:8} ({spacy.explain(ent.label_)})")
        print()

except OSError:
    print("Run: python -m spacy download en_core_web_sm")

Text: Apple Inc. was founded by Steve Jobs, Steve Wozniak, and Ron...
Entities:
  'Apple Inc.' → ORG      (Companies, agencies, institutions, etc.)
  'Steve Jobs' → PERSON   (People, including fictional)
  'Steve Wozniak' → PERSON   (People, including fictional)
  'Ronald Wayne' → PERSON   (People, including fictional)
  'Cupertino' → GPE      (Countries, cities, states)
  'California' → GPE      (Countries, cities, states)
  'April 1, 1976' → DATE     (Absolute or relative dates or periods)

Text: The United Nations headquarters is located in New York City....
Entities:
  'The United Nations' → ORG      (Companies, agencies, institutions, etc.)
  'New York City' → GPE      (Countries, cities, states)
  'António Guterres' → PERSON   (People, including fictional)

Text: Tesla reported revenue of $25.1 billion in Q3 2023. Elon Mus...
Entities:
  '$25.1 billion' → MONEY    (Monetary values, including unit)
  'Q3 2023' → DATE     (Absolute or relative dates or periods)
  'Elon Musk' → PERS

In [2]:
# ============================================================
# NER WITH HUGGING FACE
# ============================================================
try:
    from transformers import pipeline, AutoTokenizer, AutoModelForTokenClassification
    
    # Use pre-trained NER model
    ner_pipeline = pipeline(
        task="ner",
        model="dbmdz/bert-large-cased-finetuned-conll03-english",
        aggregation_strategy="simple"  # Merge subword tokens
    )
    
    text = "Barack Obama was born in Honolulu, Hawaii and served as the 44th President of the United States."
    results = ner_pipeline(text)
    
    print(f"Text: {text}\n")
    print("Entities found:")
    for entity in results:
        print(f"  '{entity['word']:20}' → {entity['entity_group']:8} (score: {entity['score']:.3f})")
        
except Exception as e:
    print(f"Requires model download: {e}")

[transformers] BertForTokenClassification LOAD REPORT from: dbmdz/bert-large-cased-finetuned-conll03-english
Key                      | Status     |  | 
-------------------------+------------+--+-
bert.pooler.dense.weight | UNEXPECTED |  | 
bert.pooler.dense.bias   | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.


Text: Barack Obama was born in Honolulu, Hawaii and served as the 44th President of the United States.

Entities found:
  'Barack Obama        ' → PER      (score: 0.999)
  'Honolulu            ' → LOC      (score: 0.998)
  'Hawaii              ' → LOC      (score: 1.000)
  'United States       ' → LOC      (score: 0.996)


In [3]:
# ============================================================
# CUSTOM NER TRAINING WITH spaCy
# ============================================================
import spacy
from spacy.tokens import DocBin
from spacy.util import filter_spans

# Training data format
TRAIN_DATA = [
    ("Microsoft was founded by Bill Gates and Paul Allen.",
     {"entities": [(0, 9, "ORG"), (31, 41, "PER"), (46, 56, "PER")]}),
    ("Google acquired YouTube for $1.65 billion.",
     {"entities": [(0, 6, "ORG"), (16, 23, "ORG"), (28, 41, "MONEY")]}),
    ("Amazon CEO Andy Jassy announced new headquarters in Seattle.",
     {"entities": [(0, 6, "ORG"), (11, 21, "PER"), (51, 58, "LOC")]}),
]

# Convert to spaCy v3 format
print("Custom NER Training Data Format:")
for text, annotations in TRAIN_DATA:
    print(f"\nText: {text}")
    for start, end, label in annotations['entities']:
        print(f"  '{text[start:end]}' → {label}")

print("\n--- Training Script (requires spacy>=3.0) ---")
print("""
import spacy
from spacy.tokens import DocBin

nlp = spacy.blank('en')
ner = nlp.add_pipe('ner')

# Add labels
for _, annotations in TRAIN_DATA:
    for _, _, label in annotations['entities']:
        ner.add_label(label)

# Convert to DocBin
db = DocBin()
for text, annotations in TRAIN_DATA:
    doc = nlp.make_doc(text)
    ents = [doc.char_span(s, e, label=l) for s, e, l in annotations['entities']]
    doc.ents = filter_spans(ents)
    db.add(doc)
db.to_disk('./train.spacy')

# Then train with: python -m spacy train config.cfg
""")

Custom NER Training Data Format:

Text: Microsoft was founded by Bill Gates and Paul Allen.
  'Microsoft' → ORG
  'ates and P' → PER
  'llen.' → PER

Text: Google acquired YouTube for $1.65 billion.
  'Google' → ORG
  'YouTube' → ORG
  '$1.65 billion' → MONEY

Text: Amazon CEO Andy Jassy announced new headquarters in Seattle.
  'Amazon' → ORG
  'Andy Jassy' → PER
  ' Seattl' → LOC

--- Training Script (requires spacy>=3.0) ---

import spacy
from spacy.tokens import DocBin

nlp = spacy.blank('en')
ner = nlp.add_pipe('ner')

# Add labels
for _, annotations in TRAIN_DATA:
    for _, _, label in annotations['entities']:
        ner.add_label(label)

# Convert to DocBin
db = DocBin()
for text, annotations in TRAIN_DATA:
    doc = nlp.make_doc(text)
    ents = [doc.char_span(s, e, label=l) for s, e, l in annotations['entities']]
    doc.ents = filter_spans(ents)
    db.add(doc)
db.to_disk('./train.spacy')

# Then train with: python -m spacy train config.cfg



In [4]:
# ============================================================
# IOB TAGGING SCHEME DEMONSTRATION
# ============================================================
def iob_to_entities(tokens, tags):
    """Convert IOB tags to entity spans"""
    entities = []
    current_entity = []
    current_type = None
    
    for token, tag in zip(tokens, tags):
        if tag.startswith('B-'):
            if current_entity:
                entities.append((' '.join(current_entity), current_type))
            current_entity = [token]
            current_type = tag[2:]
        elif tag.startswith('I-') and current_entity:
            current_entity.append(token)
        else:  # O tag
            if current_entity:
                entities.append((' '.join(current_entity), current_type))
                current_entity = []
                current_type = None
    
    if current_entity:
        entities.append((' '.join(current_entity), current_type))
    return entities

# Example
tokens = ["Barack", "Obama", "visited", "New", "York", "City", "in", "2023"]
iob_tags = ["B-PER", "I-PER", "O", "B-LOC", "I-LOC", "I-LOC", "O", "B-DATE"]

print("IOB Tagging:")
print(f"{'Token':<12} {'IOB Tag':<10}")
print("-" * 22)
for tok, tag in zip(tokens, iob_tags):
    print(f"{tok:<12} {tag:<10}")

entities = iob_to_entities(tokens, iob_tags)
print("\nExtracted Entities:")
for entity, etype in entities:
    print(f"  '{entity}' → {etype}")

IOB Tagging:
Token        IOB Tag   
----------------------
Barack       B-PER     
Obama        I-PER     
visited      O         
New          B-LOC     
York         I-LOC     
City         I-LOC     
in           O         
2023         B-DATE    

Extracted Entities:
  'Barack Obama' → PER
  'New York City' → LOC
  '2023' → DATE


In [5]:
# ============================================================
# RELATION EXTRACTION (Rule-based with spaCy)
# ============================================================
try:
    nlp = spacy.load('en_core_web_sm')
    
    def extract_relations(text):
        """Simple rule-based relation extraction using dependency parsing"""
        doc = nlp(text)
        relations = []
        
        for token in doc:
            # Pattern: SUBJECT → verb → OBJECT
            if token.dep_ in ('nsubj', 'nsubjpass') and token.head.pos_ == 'VERB':
                subject = token
                verb = token.head
                # Find object of the verb
                for child in verb.children:
                    if child.dep_ in ('dobj', 'pobj', 'attr'):
                        relations.append({
                            'subject': subject.text,
                            'relation': verb.lemma_,
                            'object': child.text
                        })
        return relations
    
    test_texts = [
        "Elon Musk founded Tesla in 2003.",
        "Amazon acquired Whole Foods for $13.7 billion.",
        "Steve Jobs launched the iPhone in 2007."
    ]
    
    for text in test_texts:
        relations = extract_relations(text)
        print(f"Text: {text}")
        for rel in relations:
            print(f"  ({rel['subject']}, {rel['relation']}, {rel['object']})")
        print()
        
except OSError:
    print("Run: python -m spacy download en_core_web_sm")

Text: Elon Musk founded Tesla in 2003.
  (Musk, found, Tesla)

Text: Amazon acquired Whole Foods for $13.7 billion.
  (Amazon, acquire, Foods)

Text: Steve Jobs launched the iPhone in 2007.
  (Jobs, launch, iPhone)



## Additional Learning Resources

### Papers
- [BiLSTM-CRF: Bidirectional LSTM-CRF Models for Sequence Labeling](https://arxiv.org/abs/1508.01991) Huang et al., 2015
- [BERT for NER: Named Entity Recognition with BERT](https://arxiv.org/abs/1910.11476) Devlin et al.
- [SpanBERT: Improving Pre-training by Representing and Predicting Spans](https://arxiv.org/abs/1907.10529)
- [Survey on NER](https://arxiv.org/abs/1812.09449) Li et al., 2020
- [REBEL: Relation Extraction By End-to-end Language generation](https://arxiv.org/abs/2110.08458)

### Tools & Libraries
- [spaCy NER Training](https://spacy.io/usage/training) Official spaCy NER guide
- [Hugging Face Token Classification](https://huggingface.co/docs/transformers/tasks/token_classification)
- [Flair NLP](https://github.com/flairNLP/flair) State-of-the-art NER library
- [GLiNER](https://github.com/urchade/GLiNER) Generalist model for NER
- [Prodigy](https://prodi.gy/) Annotation tool for NER

### Datasets
- [CoNLL-2003](https://www.clips.uantwerpen.be/conll2003/ner/) Standard NER benchmark
- [OntoNotes 5.0](https://catalog.ldc.upenn.edu/LDC2013T19) Large NER dataset
- [WNUT-17](https://noisy-text.github.io/2017/emerging-rare-entity-task.html) Emerging entities